# 07. 게임 모멘텀 피처 + 규칙(과거 수준 추정) 개선

검증: 07-26~07-30 학습 → 08-25 평가 (06과 동일). 기준선: LR+규칙 0.4299.

In [1]:
import os, glob, platform, warnings
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = {'Windows': 'Malgun Gothic', 'Darwin': 'AppleGothic'}.get(platform.system(), 'NanumGothic')
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_columns', 50)

RAW = '../data/raw/epoch_data'
D = '2026-09-01'          # 본선 기준일
FUT = 26                  # 본선 정답 구간 길이 (일)
SEED = 42

con = duckdb.connect()
for p in sorted(glob.glob(f'{RAW}/*.csv')):
    name = os.path.splitext(os.path.basename(p))[0]
    con.sql(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_csv('{p}', header=true, auto_detect=true)")

con.sql("SELECT MIN(published_at), MAX(published_at), COUNT(*) FROM video_5d_views").df()

,min(published_at),max(published_at),count_star()
0,2026-07-10 08:39:51,2026-08-31 23:59:27,31524


In [2]:
def make_features(Dref):
    f = con.sql(f"""
    WITH lv AS (
        SELECT channel_id, view_5d, like_5d, comment_5d,
               DATE_DIFF('hour', published_at, TIMESTAMP '{Dref}') / 24.0 AS days_ago
        FROM video_5d_views
        WHERE is_shorts = 0 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '{Dref}'
    ),
    v5 AS (
        SELECT channel_id,
               COUNT(*)                                        AS n_long_5d,
               MEDIAN(view_5d)                                 AS past_med,
               MEDIAN(view_5d) FILTER (WHERE days_ago < 7)     AS med_7,
               MEDIAN(view_5d) FILTER (WHERE days_ago < 14)    AS med_14,
               MEDIAN(view_5d) FILTER (WHERE days_ago >= 14)   AS med_old,
               MEDIAN(like_5d / NULLIF(view_5d, 0))            AS like_rate,
               MEDIAN(comment_5d / NULLIF(view_5d, 0))         AS comment_rate,
               STDDEV(LN(1 + view_5d))                         AS log_view_std,
               QUANTILE_CONT(LN(1 + view_5d), 0.1)             AS log_q10,
               QUANTILE_CONT(LN(1 + view_5d), 0.9)             AS log_q90,
               REGR_SLOPE(LN(1 + view_5d), -days_ago)          AS trend_slope,
               MIN(days_ago)                                   AS days_since_meas
        FROM lv GROUP BY 1
    ),
    sv AS (
        SELECT channel_id, MEDIAN(view_5d) AS short_med
        FROM video_5d_views
        WHERE is_shorts = 1 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '{Dref}'
        GROUP BY 1
    ),
    up AS (
        SELECT channel_id,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 7 DAY)  AS n_long_7,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_long_14,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 28 DAY) AS n_long_28,
               COUNT(*) FILTER (WHERE is_shorts = 1 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_short_14,
               AVG(duration_sec) FILTER (WHERE is_shorts = 0)                                                  AS avg_dur,
               DATE_DIFF('hour', MAX(published_at) FILTER (WHERE is_shorts = 0), TIMESTAMP '{Dref}') / 24.0  AS days_since_long
        FROM videos WHERE published_at < TIMESTAMP '{Dref}' GROUP BY 1
    ),
    games AS (
        SELECT v.channel_id, COUNT(DISTINCT vg.app_id) AS n_games
        FROM videos v LEFT JOIN video_games vg USING (video_id)
        WHERE v.published_at < TIMESTAMP '{Dref}' GROUP BY 1
    )
    SELECT * FROM v5 LEFT JOIN sv USING (channel_id) LEFT JOIN up USING (channel_id)
                     LEFT JOIN games USING (channel_id)
    """).df()

    lp = np.log1p(f.past_med)
    f['log_past_med']      = lp
    f['mom_7']             = np.log1p(f.med_7) - lp
    f['mom_14']            = np.log1p(f.med_14) - lp
    f['mom_old']           = np.log1p(f.med_14) - np.log1p(f.med_old)
    f['short_ratio']       = f.n_short_14 / (f.n_short_14 + f.n_long_14).replace(0, np.nan)
    f['log_short_med_rel'] = np.log1p(f.short_med) - lp
    f['has_game']          = (f.n_games > 0).astype(int)
    f['log_n_long_5d']     = np.log1p(f.n_long_5d)
    return f

FEATURES = ['log_past_med', 'n_long_5d', 'like_rate', 'n_long_14', 'n_short_14', 'short_ratio', 'days_since_long',
            'mom_14', 'log_view_std', 'n_games', 'has_game', 'mom_7', 'mom_old', 'comment_rate',
            'log_q10', 'log_q90', 'trend_slope', 'days_since_meas', 'n_long_7', 'n_long_28', 'avg_dur',
            'log_short_med_rel', 'log_n_long_5d']
print(len(FEATURES), 'features')
make_features(D)[['channel_id'] + FEATURES].head()

23 features


,channel_id,log_past_med,n_long_5d,like_rate,n_long_14,n_short_14,short_ratio,days_since_long,mom_14,log_view_std,n_games,has_game,mom_7,mom_old,comment_rate,log_q10,log_q90,trend_slope,days_since_meas,n_long_7,n_long_28,avg_dur,log_short_med_rel,log_n_long_5d
0,UCim7oePrB5hdfhcYwgUY6Jw,9.528467,22,0.005348,18,0,0.000000,0.458333,0.225331,0.265518,0,0,0.485085,0.244353,0.001496,9.347619,10.006952,0.004302,0.458333,10,35,422.672727,0.249174,3.135494
1,UCpTNtH8yc1Kdl_YFfsQsTYA,9.255409,49,0.002676,14,3,0.176471,1.458333,0.239508,1.470738,0,0,0.687683,0.239508,0.000000,8.031634,10.285952,0.024276,1.458333,8,32,9182.810345,1.164608,3.912023
2,UCos3GN2gX0wLcfuhOAiMotQ,10.477851,17,0.005466,6,1,0.142857,2.708333,-0.392605,0.610106,0,0,-0.223883,-0.576427,0.000954,9.833813,11.088090,-0.022974,2.708333,3,12,120.285714,1.325600,2.890372
3,UCz5LAkkX66OAW-y_WiWf9vA,4.532599,45,0.015038,19,8,0.296296,1.333333,0.246524,1.289942,0,0,1.714476,0.279314,0.000000,3.774058,6.544305,0.045326,1.333333,10,33,7813.352941,5.173290,3.828641
4,UCIxcZWLQpP-mUZC6_Uljc_Q,9.308646,31,0.004278,8,2,0.200000,0.416667,0.120388,0.423743,0,0,0.568908,0.145629,0.002894,8.750208,9.816458,0.012273,0.416667,4,19,350.153846,1.130369,3.465736


In [3]:
def make_labels(Dref, fut=FUT):
    g = con.sql(f"""
        WITH agg AS (
          SELECT channel_id,
                 MEDIAN(view_5d) FILTER (WHERE published_at < TIMESTAMP '{Dref}') AS past_score,
                 COUNT(view_5d)  FILTER (WHERE published_at < TIMESTAMP '{Dref}') AS n_past,
                 COALESCE(MEDIAN(view_5d) FILTER (
                     WHERE published_at >= TIMESTAMP '{Dref}'
                       AND published_at <  TIMESTAMP '{Dref}' + INTERVAL {fut} DAY), 0) AS future_score
          FROM video_5d_views
          WHERE is_shorts = 0 AND view_5d IS NOT NULL
          GROUP BY 1
        ),
        scored AS (
          SELECT channel_id, LN(1 + future_score) - LN(1 + past_score) AS growth
          FROM agg WHERE n_past >= 3 AND past_score >= 100
        )
        SELECT channel_id,
               CASE WHEN growth >= QUANTILE_CONT(growth, 0.8) OVER () THEN 1 ELSE 0 END AS target
        FROM scored
    """).df()
    g['ref_date'] = Dref
    return g


def attach_features(labels):
    return pd.concat([labels[labels.ref_date == d].merge(make_features(d), on='channel_id', how='left')
                      for d in sorted(labels.ref_date.unique())], ignore_index=True)


# 우리가 만든 26일 라벨 — 과거 이력이 2주 이상 쌓인 07-26 ~ 08-05
OWN_DATES = pd.date_range('2026-07-26', '2026-08-05', freq='D').strftime('%Y-%m-%d').tolist()
own = attach_features(pd.concat([make_labels(d) for d in OWN_DATES], ignore_index=True))

# 공식 라벨 (18일 창)
tl = pd.read_csv(f'{RAW}/train_labels.csv')
tl['channel_id'] = tl.row_id.str.rsplit('_', n=1).str[0]
tl['ref_date']   = tl.row_id.str.rsplit('_', n=1).str[1]
official = attach_features(tl[['channel_id', 'ref_date', 'target']])

summary = pd.concat([own, official]).groupby('ref_date').target.agg(['size', 'mean']).round(3)
summary['window'] = ['26일(자체)'] * len(OWN_DATES) + ['18일(공식)'] * 2
summary

,size,mean,window
ref_date,,,
2026-07-26,349,0.201,26일(자체)
2026-07-27,389,0.201,26일(자체)
2026-07-28,414,0.200,26일(자체)
2026-07-29,430,0.200,26일(자체)
2026-07-30,448,0.201,26일(자체)
2026-07-31,479,0.200,26일(자체)
2026-08-01,494,0.200,26일(자체)
2026-08-02,509,0.200,26일(자체)
2026-08-03,523,0.201,26일(자체)


In [4]:
def logreg(C=0.1):
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('sc', StandardScaler()),
                     ('m', LogisticRegression(max_iter=2000, C=C, class_weight='balanced'))])

def rank01(x):
    x = pd.Series(np.asarray(x, dtype=float))
    return rankdata(x.fillna(x.median())) / len(x)

def blend(proba, df):
    """모델 확률 순위 + 과거 중앙값 역순위의 평균 (PR-AUC는 순위만 봄)"""
    return (rank01(proba) + rank01(-df.log_past_med)) / 2



In [5]:
for t in ['video_games','app_snapshots','app_reviews','apps','games']:
    try:
        print('=====', t)
        print(con.sql(f"DESCRIBE {t}").df()[['column_name','column_type']].T.to_string())
        print(con.sql(f"SELECT COUNT(*) n FROM {t}").df().iloc[0,0], '행')
        print(con.sql(f"SELECT * FROM {t} LIMIT 3").df().to_string())
    except Exception as e:
        print(t, 'X', str(e)[:80])
print(con.sql("SHOW TABLES").df().name.tolist())

===== video_games
                    0       1              2             3           4
column_name  video_id  app_id  matched_alias  match_method  confidence
column_type   VARCHAR  BIGINT        VARCHAR       VARCHAR      DOUBLE
1826 행
      video_id   app_id matched_alias match_method  confidence
0  --0LsFjsPNs   271590         gta 5        title        0.85
1  -04xqnY2bbA   578080        배틀그라운드        title        0.85
2  -0gqp7AFBBg  1091500         사이버펑크        title        0.85
===== app_snapshots
                 0             1                2            3                 4             5                  6              7               8               9               10               11
column_name  app_id  collected_at  current_players  price_final  discount_percent  review_score  review_score_desc  total_reviews  total_positive  total_negative  positive_ratio  recommendations
column_type  BIGINT     TIMESTAMP           BIGINT       BIGINT            BIGINT        BIGINT     

29262 행
    app_id        collected_at  current_players  price_final  discount_percent  review_score review_score_desc  total_reviews  total_positive  total_negative  positive_ratio  recommendations
0  1245620 2026-08-07 21:00:01            29720      6480000                 0             8     Very Positive        1147255         1067827           79428          0.9308           826295
1      570 2026-08-07 21:00:05           687192         <NA>              <NA>             8     Very Positive        2753211         2219107          534104          0.8060            14378
2      730 2026-08-07 21:00:08           999397         <NA>              <NA>             8     Very Positive        9775017         8405765         1369252          0.8599          5209660
===== app_reviews


                            0       1             2         3         4         5            6                    7              8               9                  10                           11                      12                 13             14           15
column_name  recommendation_id  app_id  collected_at  language  voted_up  votes_up  votes_funny  weighted_vote_score  comment_count  steam_purchase  received_for_free  written_during_early_access  playtime_at_review_min  timestamp_created  review_length  review_text
column_type             BIGINT  BIGINT     TIMESTAMP   VARCHAR    BIGINT    BIGINT       BIGINT               DOUBLE         BIGINT          BIGINT             BIGINT                       BIGINT                  BIGINT             BIGINT         BIGINT      VARCHAR
148157 행


   recommendation_id  app_id        collected_at   language  voted_up  votes_up  votes_funny  weighted_vote_score  comment_count  steam_purchase  received_for_free  written_during_early_access  playtime_at_review_min  timestamp_created  review_length review_text
0          231952689     730 2026-08-03 09:02:26  ukrainian         1         0            0                  0.5              0               1                  0                            0                     414         1785741454              9        None
1          231952602     730 2026-08-03 09:02:26  ukrainian         1         0            0                  0.5              0               1                  0                            0                    1488         1785741318             20        None
2          231952578     730 2026-08-03 09:02:26   schinese         1         0            0                  0.5              0               1                  0                            0                   

## 1. 게임 계열 테이블의 수집 기간 (기준일 이전에 쓸 수 있는가)

In [6]:
q = {
 'app_snapshots': "SELECT MIN(collected_at) a, MAX(collected_at) b, COUNT(DISTINCT app_id) apps, COUNT(*) n FROM app_snapshots",
 'app_reviews (timestamp_created)': "SELECT TO_TIMESTAMP(MIN(timestamp_created)) a, TO_TIMESTAMP(MAX(timestamp_created)) b, COUNT(DISTINCT app_id) apps, COUNT(*) n FROM app_reviews",
 'app_reviews (collected_at)': "SELECT MIN(collected_at) a, MAX(collected_at) b, COUNT(DISTINCT app_id) apps, COUNT(*) n FROM app_reviews",
 'igdb_game_snapshots': "SELECT MIN(collected_at) a, MAX(collected_at) b, COUNT(*) n FROM igdb_game_snapshots",
 'video_topics': "SELECT COUNT(*) n, COUNT(DISTINCT video_id) v FROM video_topics",
 'video_igdb_games': "SELECT COUNT(*) n, COUNT(DISTINCT video_id) v FROM video_igdb_games",
 'review_features': "SELECT COUNT(*) n FROM review_features",
}
for k, s in q.items():
    print('==', k); print(con.sql(s).df().to_string())
print(con.sql("DESCRIBE igdb_game_snapshots").df()[['column_name']].T.to_string())
print(con.sql("DESCRIBE review_features").df()[['column_name']].T.to_string())
print(con.sql("DESCRIBE video_topics").df()[['column_name']].T.to_string())
print(con.sql("SELECT * FROM igdb_game_snapshots LIMIT 3").df().to_string())

== app_snapshots
                    a                   b  apps      n
0 2026-08-03 07:34:20 2026-08-31 23:02:29    47  29262
== app_reviews (timestamp_created)


                          a                         b  apps       n
0 2026-06-23 11:00:11+09:00 2026-09-01 03:21:08+09:00    47  148157
== app_reviews (collected_at)
                    a                   b  apps       n
0 2026-08-03 07:34:20 2026-08-31 18:33:42    47  148157
== igdb_game_snapshots


                    a                   b    n
0 2026-08-17 10:49:01 2026-08-31 05:00:01  992
== video_topics
       n     v
0  14840  6118
== video_igdb_games
       n      v
0  10386  10302
== review_features
      n
0  2369


                   0             1       2             3                  4                        5             6                   7        8      9
column_name  igdb_id  collected_at  rating  rating_count  aggregated_rating  aggregated_rating_count  total_rating  total_rating_count  follows  hypes
                  0             1                    2                  3                       4                       5                          6                     7              8
column_name  app_id  collected_at  num_reviews_sampled  avg_review_length  positive_ratio_sampled  avg_playtime_at_review  early_access_review_ratio  steam_purchase_ratio  top_languages
                    0               1
column_name  video_id  topic_category


   igdb_id        collected_at  rating  rating_count  aggregated_rating  aggregated_rating_count  total_rating  total_rating_count follows  hypes
0      115 2026-08-17 11:20:39  75.515          1012             84.000                        3        79.758                1015    None   <NA>
1     8173 2026-08-17 11:20:39  81.791          1170             89.083                       24        85.437                1194    None     41
2   126459 2026-08-17 11:20:39  71.617           502             78.333                        6        74.975                 508    None     30


### 결론: 게임 모멘텀 피처는 사용 불가
`app_snapshots`는 08-03부터, `igdb_game_snapshots`는 08-17부터만 수집되고 대상도 47개 게임뿐이다. 학습 기준일(07-26~08-05)에는 기준일 이전 데이터가 없어 구독자 피처와 같은 문제(학습/예측 분포 불일치)가 생긴다 → 제외.

## 2. 규칙(과거 수준 추정) 개선
라벨은 `log(미래) − log(과거 중앙값)`. 규칙은 학습이 필요 없으므로 **라벨이 있는 모든 기준일(자체 FUT=26 11개 + 공식 08-11, 08-25)** 에서 평가해 단일 기준일 노이즈를 줄인다.
점수 = `a·L − log_past_med` (L = 미래 수준 추정치). a=0이면 기존 규칙. L 후보: 최근 k편 중앙값, 로그 평균, 상위 분위수 등.

In [7]:
def level_feats(Dref):
    return con.sql(f"""
    WITH lv AS (
        SELECT channel_id, LN(1 + view_5d) AS l, published_at,
               ROW_NUMBER() OVER (PARTITION BY channel_id ORDER BY published_at DESC) AS rn
        FROM video_5d_views
        WHERE is_shorts = 0 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '{Dref}'
    )
    SELECT channel_id,
           AVG(l) AS L_mean,
           MEDIAN(l) FILTER (WHERE rn <= 3)  AS L_med3,
           MEDIAN(l) FILTER (WHERE rn <= 5)  AS L_med5,
           MEDIAN(l) FILTER (WHERE rn <= 10) AS L_med10,
           QUANTILE_CONT(l, 0.75) AS L_q75,
           QUANTILE_CONT(l, 0.25) AS L_q25,
           MAX(l) AS L_max
    FROM lv GROUP BY 1
    """).df()

labeled = pd.concat([own, official], ignore_index=True)
lf = pd.concat([level_feats(d).assign(ref_date=d) for d in sorted(labeled.ref_date.unique())], ignore_index=True)
lab = labeled.merge(lf, on=['channel_id', 'ref_date'], how='left')
dates = sorted(lab.ref_date.unique())
print(len(lab), '행,', len(dates), '개 기준일')

def eval_rule(score_fn):
    s = [average_precision_score(g.target, score_fn(g)) for _, g in lab.groupby('ref_date')]
    last = lab[lab.ref_date == '2026-08-25']
    return np.mean(s), average_precision_score(last.target, score_fn(last))

base_m, base_l = eval_rule(lambda g: -g.log_past_med.fillna(g.log_past_med.median()))
print(f'기존 규칙: 13일 평균 {base_m:.4f} | 08-25 {base_l:.4f}')

rows = []
for L in ['L_mean', 'L_med3', 'L_med5', 'L_med10', 'L_q75', 'L_q25', 'L_max']:
    for a in [0.1, 0.2, 0.3, 0.5, 0.8]:
        f = lambda g, L=L, a=a: (a * g[L].fillna(g[L].median()) - g.log_past_med.fillna(g.log_past_med.median()))
        m, l = eval_rule(f)
        rows.append((L, a, round(m, 4), round(l, 4)))
rr = pd.DataFrame(rows, columns=['L', 'a', '13일평균', '08-25'])
print(rr.pivot(index='L', columns='a', values='13일평균'))
print(rr.sort_values('13일평균', ascending=False).head(5))

6369 행, 13 개 기준일
기존 규칙: 13일 평균 0.2962 | 08-25 0.4039


a           0.1     0.2     0.3     0.5     0.8
L                                              
L_max    0.2967  0.3021  0.3053  0.3166  0.3100
L_mean   0.2956  0.2945  0.2965  0.2999  0.3200
L_med10  0.2962  0.2971  0.2976  0.2961  0.2944
L_med3   0.2997  0.3025  0.3038  0.3083  0.3167
L_med5   0.2979  0.2999  0.3017  0.3014  0.3059
L_q25    0.2970  0.2964  0.2956  0.2972  0.2878
L_q75    0.2962  0.2978  0.2998  0.3056  0.3356
         L    a   13일평균   08-25
24   L_q75  0.8  0.3356  0.4259
4   L_mean  0.8  0.3200  0.4325
9   L_med3  0.8  0.3167  0.4207
33   L_max  0.5  0.3166  0.4290
34   L_max  0.8  0.3100  0.3637


### 2.1 결과 관찰 + a 범위 확장
`a·L_q75 − log_past_med`는 `a·(q75 − med) − (1−a)·med` 로 쓸 수 있어, **분위수 간 퍼짐(가끔 크게 터지는 정도)** 과 낮은 과거 수준을 함께 보는 점수다. a=0.8이 격자 끝이라 범위와 분위수를 넓혀 본다.

In [8]:
for g_ in lab.groupby('ref_date'): pass
lab['L_q60'] = np.nan
# 분위수 추가 계산
def q_feats(Dref):
    return con.sql(f"""
    WITH lv AS (SELECT channel_id, LN(1 + view_5d) AS l FROM video_5d_views
                WHERE is_shorts = 0 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '{Dref}')
    SELECT channel_id, QUANTILE_CONT(l, 0.6) AS L_q60, QUANTILE_CONT(l, 0.9) AS L_q90 FROM lv GROUP BY 1
    """).df()
qf = pd.concat([q_feats(d).assign(ref_date=d) for d in dates], ignore_index=True)
lab = lab.drop(columns=['L_q60']).merge(qf, on=['channel_id', 'ref_date'], how='left')

rows = []
for L in ['L_q60', 'L_q75', 'L_q90', 'L_max', 'L_mean']:
    for a in [0.6, 0.8, 0.9, 1.0, 1.2, 1.5]:
        f = lambda g, L=L, a=a: (a * g[L].fillna(g[L].median()) - g.log_past_med.fillna(g.log_past_med.median()))
        m, l = eval_rule(f)
        rows.append((L, a, round(m, 4), round(l, 4)))
rr2 = pd.DataFrame(rows, columns=['L', 'a', '13일평균', '08-25'])
print(rr2.pivot(index='L', columns='a', values='13일평균'))

# 날짜별 비교: 기존 규칙 vs 후보
def per_date(f):
    return pd.Series({d: average_precision_score(g.target, f(g)) for d, g in lab.groupby('ref_date')})
base_f = lambda g: -g.log_past_med.fillna(g.log_past_med.median())
best = rr2.sort_values('13일평균', ascending=False).iloc[0]
print('최고 후보:', best.L, best.a)
cand_f = lambda g: best.a * g[best.L].fillna(g[best.L].median()) - g.log_past_med.fillna(g.log_past_med.median())
cmp = pd.DataFrame({'기존': per_date(base_f), '후보': per_date(cand_f)}).round(4)
cmp['차이'] = (cmp['후보'] - cmp['기존']).round(4)
print(cmp)
print('후보가 이긴 날짜:', (cmp['차이'] > 0).sum(), '/', len(cmp))

a          0.6     0.8     0.9     1.0     1.2     1.5
L                                                     
L_max   0.3201  0.3100  0.3014  0.2879  0.2593  0.2279
L_mean  0.3026  0.3200  0.3359  0.3185  0.2102  0.1726
L_q60   0.3031  0.3178  0.3247  0.2971  0.2003  0.1752
L_q75   0.3133  0.3356  0.3379  0.3169  0.2509  0.2067
L_q90   0.3208  0.3305  0.3253  0.3114  0.2676  0.2246
최고 후보: L_q75 0.9
                기존      후보      차이
2026-07-26  0.2995  0.3001  0.0006
2026-07-27  0.2929  0.3155  0.0226
2026-07-28  0.2470  0.3374  0.0904
2026-07-29  0.2725  0.3526  0.0801
2026-07-30  0.2699  0.2986  0.0287
2026-07-31  0.2893  0.3061  0.0168
2026-08-01  0.3123  0.3355  0.0232
2026-08-02  0.3062  0.3324  0.0262
2026-08-03  0.3011  0.3374  0.0363
2026-08-04  0.2921  0.3874  0.0953
2026-08-05  0.2728  0.3504  0.0776
2026-08-11  0.2914  0.3445  0.0531
2026-08-25  0.4039  0.3950 -0.0089
후보가 이긴 날짜: 12 / 13


### 2.2 LR과 섞은 blend (시간 분리: 07-26~07-30 학습 → 08-25 평가)
규칙 후보 `a·L_q75 − log_past_med` 를 기존 규칙 자리에 넣어 LR과 rank 평균. 08-25는 FUT=18 기준일이라 본선(FUT=26) 조건과 다르다는 점에 유의.

In [9]:
trm = lab[(lab.ref_date >= '2026-07-26') & (lab.ref_date <= '2026-07-30') & lab.ref_date.isin(own.ref_date.unique())]
vam = lab[lab.ref_date == '2026-08-25']
Fm = FEATURES
p_lr = logreg(0.1).fit(trm[Fm], trm.target).predict_proba(vam[Fm])[:, 1]
ym = vam.target

def rule_score(g, a, L='L_q75'):
    return a * g[L].fillna(g[L].median()) - g.log_past_med.fillna(g.log_past_med.median())

out = [('LR 단독', average_precision_score(ym, p_lr))]
for a in [0, 0.5, 0.8, 0.9]:
    r_ = rule_score(vam, a)
    out.append((f'규칙 단독 a={a}', average_precision_score(ym, r_)))
    for w in [0.5, 0.7]:
        out.append((f'LR + 규칙 a={a}, 규칙가중 {w}', average_precision_score(ym, (1 - w) * rank01(p_lr) + w * rank01(r_))))
pd.DataFrame(out, columns=['방법', 'PR-AUC(08-25)']).round(4)

,방법,PR-AUC(08-25)
0,LR 단독,0.3815
1,규칙 단독 a=0,0.4039
2,"LR + 규칙 a=0, 규칙가중 0.5",0.4299
3,"LR + 규칙 a=0, 규칙가중 0.7",0.4255
4,규칙 단독 a=0.5,0.4171
5,"LR + 규칙 a=0.5, 규칙가중 0.5",0.4443
6,"LR + 규칙 a=0.5, 규칙가중 0.7",0.4490
7,규칙 단독 a=0.8,0.4259
8,"LR + 규칙 a=0.8, 규칙가중 0.5",0.4421
9,"LR + 규칙 a=0.8, 규칙가중 0.7",0.4474


## 3. 제출 후보 생성: LR + 새 규칙
- 규칙: `0.8·L_q75 − log_past_med` (13개 기준일 평균 0.296 → 약 0.336, 12/13일 개선)
- 점수: `0.5·rank(LR 확률) + 0.5·rank(규칙)`. LR은 05와 동일(C=0.1, 학습 6,369행)
- 비교 기준: 기존 제출(규칙 a=0) 리더보드 0.4554

In [10]:
A, W = 0.8, 0.5
train_all = pd.concat([own, official], ignore_index=True)
final_model = logreg(0.1).fit(train_all[FEATURES], train_all.target)

sample = pd.read_csv(f'{RAW}/sample_submission.csv', encoding='utf-8-sig')
sample['channel_id'] = sample.row_id.str.rsplit('_', n=1).str[0]
test = (sample[['row_id', 'channel_id']]
        .merge(make_features(D), on='channel_id', how='left')
        .merge(level_feats(D), on='channel_id', how='left'))
print('결측 (log_past_med, L_q75):', test.log_past_med.isna().sum(), test.L_q75.isna().sum())

proba = final_model.predict_proba(test[FEATURES])[:, 1]
test['prediction'] = (1 - W) * rank01(proba) + W * rank01(rule_score(test, A))
sub = sample[['row_id']].merge(test[['row_id', 'prediction']], on='row_id', how='left')

assert list(sub.columns) == ['row_id', 'prediction'] and len(sub) == len(sample) == 694
assert set(sub.row_id) == set(sample.row_id) and sub.row_id.is_unique
assert sub.prediction.notna().all() and np.isfinite(sub.prediction).all() and sub.prediction.between(0, 1).all()

FILE = '../submissions/submit_05_logreg_q75rule_a08.csv'
sub.to_csv(FILE, index=False, encoding='utf-8')

# 기존 제출과 순위 상관 (너무 다르면 의심, 너무 같으면 제출 가치 없음)
old = pd.read_csv('../submissions/submit_04_logreg_rankblend_fut26.csv')
print('저장:', FILE, '| 기존 제출과 스피어만 상관:', round(sub.prediction.corr(old.set_index('row_id').loc[sub.row_id, 'prediction'].reset_index(drop=True), method='spearman'), 4))

결측 (log_past_med, L_q75): 0 0
저장: ../submissions/submit_05_logreg_q75rule_a08.csv | 기존 제출과 스피어만 상관: 0.9653


## 4. IGDB 출시일 피처 탐색
채널이 다루는 게임이 **예측 구간(D~D+26) 안에 출시**되거나 **최근 출시**됐다면 조회수가 오를 가능성. IGDB 정적 정보(출시일)는 기준일 이전에 알 수 있어 시간 불일치가 없다.

In [11]:
for t in ['igdb_games', 'igdb_game_map', 'video_igdb_games', 'game_app_map']:
    print('=====', t, con.sql(f"SELECT COUNT(*) FROM {t}").df().iloc[0, 0], '행')
    print(con.sql(f"SELECT * FROM {t} LIMIT 3").df().T.to_string())

===== igdb_games 58 행
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             0                                                                                                                                                             1                                                                                                                                                                                      

In [12]:
print(con.sql("""SELECT name, first_release_date, status FROM igdb_games
                 WHERE first_release_date >= '2025-06-01' ORDER BY first_release_date""").df().to_string())
print(con.sql("SELECT MIN(first_release_date) mn, MAX(first_release_date) mx, COUNT(*) n FROM igdb_games").df())
print(con.sql("SELECT release_date, name FROM apps WHERE coming_soon=1 OR release_date LIKE '%2026%' LIMIT 10").df())

       name first_release_date  status
0   Lineage         2026-02-01       4
1  Palworld         2026-07-10    <NA>
2   Valheim         2026-09-09    <NA>
          mn         mx   n
0 1998-03-31 2026-09-09  58
  release_date            name
0  2026년 9월 9일         Valheim
1  2026년 7월 9일  Palworld / 팰월드


### 4 결론: IGDB 출시일 피처 사용 불가
`igdb_games`는 58개 게임뿐이고 대부분 오래된 작품이다. 2025-06 이후 출시작은 Lineage(02-01), Palworld(07-10), Valheim(09-09) 정도로, 예측 구간(09-01~09-27)에 들어가는 것은 Valheim 하나뿐이라 학습/일반화가 불가능 → 제외.

In [13]:
print(FEATURES)
print(lab[["n_long_28","n_long_5d","log_view_std","log_past_med"]].describe().T)

['log_past_med', 'n_long_5d', 'like_rate', 'n_long_14', 'n_short_14', 'short_ratio', 'days_since_long', 'mom_14', 'log_view_std', 'n_games', 'has_game', 'mom_7', 'mom_old', 'comment_rate', 'log_q10', 'log_q90', 'trend_slope', 'days_since_meas', 'n_long_7', 'n_long_28', 'avg_dur', 'log_short_med_rel', 'log_n_long_5d']
               count       mean        std       min       25%        50%  \
n_long_28     6369.0  17.735594  20.797581  0.000000  8.000000  12.000000   
n_long_5d     6369.0  12.580154  17.216536  2.000000  5.000000   8.000000   
log_view_std  6369.0   0.775319   0.664174  0.031215  0.382735   0.598104   
log_past_med  6369.0   9.334491   1.860871  4.369448  8.213653   9.583592   

                    75%         max  
n_long_28     20.000000  218.000000  
n_long_5d     14.000000  310.000000  
log_view_std   0.936662    6.102662  
log_past_med  10.648539   13.643552  


## 5. 구조 모형 규칙 (경험적 베이즈 수축 + 꼬리 확률 + 업로드 유무)
채널 i의 영상 로그 조회수 ~ N(μᵢ, σᵢ²). 기준일마다 모집단 평균 m, 채널 간 분산 τ²를 추정한다.
- 사후: μ̂ = w·ℓ + (1−w)·m, w = τ²/(τ² + σᵢ²/nᵢ) (ℓ = log 과거 중앙값), 사후분산 v = w·σᵢ²/nᵢ
- 미래 중앙값 노이즈: 1.57·σᵢ²/M (M = 예상 업로드 수 = 일평균 업로드 × 26)
- growth ~ N(μ̂ − ℓ, v + 1.57σᵢ²/M), 점수 = P(growth > t) × (1 − P(업로드 0))
t(임계값)는 스칼라 하나만 격자로 고른다.

In [14]:
from scipy.stats import norm

def struct_score(g, t=0.5, k=3, use_p0=True, rate_days=28, fut=26, noise_mult=1.57):
    n   = g.n_long_5d.astype(float)
    l   = g.log_past_med.astype(float)
    s2p = np.nanmedian(g.log_view_std ** 2)                         # 풀링 분산
    s2  = (((n - 1) * g.log_view_std.fillna(np.sqrt(s2p)) ** 2) + k * s2p) / (n - 1 + k)   # 채널 분산 수축
    m   = l.mean()
    tau2 = max(l.var() - np.mean(s2 / n), 0.05)
    w   = tau2 / (tau2 + s2 / n)
    mu  = w * l + (1 - w) * m
    v   = w * s2 / n
    rate = (g.n_long_28.fillna(0) + 0.5) / rate_days                # 일평균 롱폼 업로드 (+0.5 평활)
    M   = np.maximum(rate * fut, 1.0)
    sd  = np.sqrt(v + noise_mult * s2 / M)
    p   = norm.sf((t - (mu - l)) / sd)
    if use_p0:
        p = p * (1 - np.exp(-rate * fut))
    return p

rows = []
for t in [0.0, 0.25, 0.5, 0.75, 1.0]:
    for use_p0 in [False, True]:
        m_, l_ = eval_rule(lambda g, t=t, u=use_p0: struct_score(g, t=t, use_p0=u))
        rows.append((t, use_p0, round(m_, 4), round(l_, 4)))
st = pd.DataFrame(rows, columns=['t', 'p0곱', '13일평균', '08-25'])
print('기존 규칙 0.2962 | q75 규칙(a=0.8) 0.3356')
st

기존 규칙 0.2962 | q75 규칙(a=0.8) 0.3356


,t,p0곱,13일평균,08-25
0,0.00,False,0.3119,0.3892
1,0.00,True,0.3047,0.3847
2,0.25,False,0.3287,0.2849
3,0.25,True,0.3273,0.2920
4,0.50,False,0.3300,0.2686
5,0.50,True,0.3288,0.2704
6,0.75,False,0.3270,0.2625
7,0.75,True,0.3266,0.2615
8,1.00,False,0.3271,0.2595
9,1.00,True,0.3269,0.2568


### 5.1 규칙 결합 + LR 피처 축소
(a) q75 규칙과 구조 모형을 rank 평균하면 13일 평균이 오르는가
(b) LR이 규칙 단독보다 낮은 이유가 피처 과다(23개)인지: 소수 피처 LR을 같은 시간 분리(07-26~07-30 → 08-25)에서 비교

In [15]:
rk = lambda s: pd.Series(np.asarray(s, float)).rank(pct=True).values
q75f  = lambda g: rule_score(g, 0.8)
strf  = lambda g: struct_score(g, t=0.5, use_p0=False)
mix = lambda g, w=0.5: (1 - w) * rk(q75f(g)) + w * rk(strf(g))
print('q75 규칙      13일평균 %.4f | 08-25 %.4f' % eval_rule(q75f))
print('구조 모형     13일평균 %.4f | 08-25 %.4f' % eval_rule(strf))
for w in [0.3, 0.5, 0.7]:
    print('결합 w=%.1f   13일평균 %.4f | 08-25 %.4f' % ((w,) + eval_rule(lambda g, w=w: mix(g, w))))

# (b) 소수 피처 LR (시간 분리)
trm2 = lab[lab.ref_date.isin(['2026-07-26', '2026-07-27', '2026-07-28', '2026-07-29', '2026-07-30'])]
vam2 = lab[lab.ref_date == '2026-08-25']
sets = {
 '전체23개':                FEATURES,
 '핵심4 (past,std,n,n_long_28)': ['log_past_med', 'log_view_std', 'n_long_5d', 'n_long_28'],
 '핵심4 + q75,mean':        ['log_past_med', 'log_view_std', 'n_long_5d', 'n_long_28', 'L_q75', 'L_mean'],
 '핵심4 + q75,mean,q90,max': ['log_past_med', 'log_view_std', 'n_long_5d', 'n_long_28', 'L_q75', 'L_mean', 'L_q90', 'L_max'],
}
res = []
for name, cols in sets.items():
    for C in [0.01, 0.1, 1.0]:
        pr = logreg(C).fit(trm2[cols], trm2.target).predict_proba(vam2[cols])[:, 1]
        res.append((name, C, round(average_precision_score(vam2.target, pr), 4),
                    round(average_precision_score(vam2.target, 0.5 * rk(pr) + 0.5 * rk(q75f(vam2))), 4)))
pd.DataFrame(res, columns=['피처셋', 'C', 'LR 단독', 'LR+q75규칙 blend'])

q75 규칙      13일평균 0.3356 | 08-25 0.4259
구조 모형     13일평균 0.3300 | 08-25 0.2686
결합 w=0.3   13일평균 0.3455 | 08-25 0.3873
결합 w=0.5   13일평균 0.3473 | 08-25 0.3573
결합 w=0.7   13일평균 0.3441 | 08-25 0.3199


,피처셋,C,LR 단독,LR+q75규칙 blend
0,전체23개,0.01,0.3806,0.4434
1,전체23개,0.10,0.3815,0.4421
2,전체23개,1.00,0.3786,0.4408
3,"핵심4 (past,std,n,n_long_28)",0.01,0.3182,0.3875
4,"핵심4 (past,std,n,n_long_28)",0.10,0.2777,0.3772
5,"핵심4 (past,std,n,n_long_28)",1.00,0.2638,0.3762
6,"핵심4 + q75,mean",0.01,0.3185,0.3883
7,"핵심4 + q75,mean",0.10,0.2929,0.3862
8,"핵심4 + q75,mean",1.00,0.2928,0.3929
9,"핵심4 + q75,mean,q90,max",0.01,0.3286,0.3952


### 5.2 정리 (이 노트북에서 시험한 것)
| 시도 | 결과 | 판단 |
|---|---|---|
| 게임 모멘텀 (Steam/IGDB 스냅샷) | 수집이 08-03/08-17 이후라 학습 기준일에 없음, 게임 47~58개 | 사용 불가 |
| **규칙 `0.8·L_q75 − log_past_med`** | 13일 평균 0.296→0.336, 12/13일 개선 (FUT=26 11일 전부), 08-25 0.404→0.426 | **채택** (submit_05) |
| 구조 모형 (경험적 베이즈+꼬리확률) | 13일 평균 0.330이나 08-25에서 0.27로 급락, 업로드 유무(p0) 곱은 이득 없음 | 불안정, 제외 |
| q75 규칙 + 구조 모형 결합 | 13일 0.347이지만 08-25는 0.357로 하락 | 08-25와 상충, 제외 |
| LR 피처 축소(4~8개) | LR 단독 0.28~0.33 < 23개 0.38 | 모멘텀/반응 등 나머지 피처가 정보를 가짐, 축소 안 함 |

**한계**: 검증은 FUT=26 자체 기준일(서로 강하게 겹침)과 FUT=18 공식 08-25 단일 기준일뿐이라, 0.01~0.02 수준 차이는 구분 불가. 구조 모형처럼 두 검증이 상충하면 채택하지 않았다.

## 6. 검증 강화: 채널 단위 GroupKFold + 이력 긴 기준일 추가
**문제**: 지금까지 모델 평가는 08-25 한 날짜(666행)뿐이라 리더보드와 어긋났다. 날짜 단위로 가르면 라벨 창이 겹쳐 시간 누수가 생긴다.
**해결**: 같은 채널의 모든 기준일 행을 한 fold에 묶는다(GroupKFold). 라벨은 같은 날짜 안의 상대 순위(상위 20%)라 시간 시장 효과는 상쇄되고, 채널 단위 누수가 없다. 모든 기준일에서 날짜별 PR-AUC를 구해 평균한다.
**추가 기준일**: 자체 FUT=18 라벨로 08-06~08-13(08-11 제외, 공식 존재)을 만들어 이력이 길고(9/1에 가까운) 공식 라벨과 같은 FUT=18인 날짜를 늘린다.

그룹 정의: `early` = 07-26~08-05 (FUT=26, 이력 짧음) / `late` = 08-06~08-13, 08-25 (FUT=18, 이력 김)

In [16]:
EXTRA18 = ['2026-08-06', '2026-08-07', '2026-08-08', '2026-08-09', '2026-08-10', '2026-08-12', '2026-08-13']
own18 = attach_features(pd.concat([make_labels(d, fut=18) for d in EXTRA18], ignore_index=True))
lf18 = pd.concat([level_feats(d).assign(ref_date=d) for d in EXTRA18], ignore_index=True)
qf18 = pd.concat([q_feats(d).assign(ref_date=d) for d in EXTRA18], ignore_index=True)
own18 = own18.merge(lf18, on=['channel_id', 'ref_date'], how='left').merge(qf18, on=['channel_id', 'ref_date'], how='left')

lab2 = pd.concat([lab, own18], ignore_index=True)
lab2['grp'] = np.where(lab2.ref_date <= '2026-08-05', 'early', 'late')
print(lab2.groupby('ref_date').target.agg(['size', 'mean']).round(3).T.to_string())
print('총', len(lab2), '행, 채널', lab2.channel_id.nunique(), '개')

ref_date  2026-07-26  2026-07-27  2026-07-28  2026-07-29  2026-07-30  2026-07-31  2026-08-01  2026-08-02  2026-08-03  2026-08-04  2026-08-05  2026-08-06  2026-08-07  2026-08-08  2026-08-09  2026-08-10  2026-08-11  2026-08-12  2026-08-13  2026-08-25
size         349.000     389.000       414.0       430.0     448.000       479.0       494.0       509.0     523.000       530.0     537.000     552.000       569.0       584.0     591.000     593.000     601.000     603.000     611.000     666.000
mean           0.201       0.201         0.2         0.2       0.201         0.2         0.2         0.2       0.201         0.2       0.201       0.201         0.2         0.2       0.201       0.201       0.201       0.201       0.201       0.201
총 10472 행, 채널 672 개


In [17]:
from sklearn.model_selection import GroupKFold

def oof_predict(df, model_fns, n_splits=5, seed=0):
    """채널 단위 GroupKFold OOF 예측. model_fns: {이름: fn(train_df, test_df)->score}"""
    ch = df.channel_id.values
    uniq = np.unique(ch); rng = np.random.default_rng(seed); rng.shuffle(uniq)
    fold_of = {c: i % n_splits for i, c in enumerate(uniq)}
    fold = np.array([fold_of[c] for c in ch])
    out = {k: np.zeros(len(df)) for k in model_fns}
    for f in range(n_splits):
        trn, tst = df[fold != f], df[fold == f]
        for k, fn in model_fns.items():
            out[k][fold == f] = fn(trn, tst)
    return pd.DataFrame(out, index=df.index)

def summarize(df, oof):
    res = {}
    for k in oof.columns:
        per = {d: average_precision_score(g.target, oof.loc[g.index, k]) for d, g in df.groupby('ref_date')}
        per = pd.Series(per)
        res[k] = {'전체평균': per.mean(), 'early평균': per[per.index <= '2026-08-05'].mean(),
                  'late평균': per[per.index > '2026-08-05'].mean(), '08-25': per['2026-08-25']}
    return pd.DataFrame(res).T.round(4)

In [18]:
import lightgbm as lgb
def gb(seed, **kw):
    p = dict(n_estimators=200, learning_rate=0.03, num_leaves=4, max_depth=3, min_child_samples=60,
             subsample=0.7, subsample_freq=1, colsample_bytree=0.7, reg_lambda=10, random_state=seed, verbose=-1)
    p.update(kw)
    return lgb.LGBMClassifier(**p)

def lr_fn(trn, tst, C=0.1, cols=None):
    cols = cols or FEATURES
    return logreg(C).fit(trn[cols], trn.target).predict_proba(tst[cols])[:, 1]

def lgb_fn(trn, tst):
    return np.mean([gb(s).fit(trn[FEATURES], trn.target).predict_proba(tst[FEATURES])[:, 1] for s in range(3)], axis=0)

rule0 = lambda g: -g.log_past_med.fillna(g.log_past_med.median())
fns = {
    'rule0 (-log_past_med)': lambda trn, tst: rule0(tst).values,
    'rule q75 a=0.8':        lambda trn, tst: rule_score(tst, 0.8).values,
    'LR':                    lr_fn,
    'LGB':                   lgb_fn,
}
oof = oof_predict(lab2, fns)
# 날짜 내 rank로 blend (같은 날짜 안 순위)
rk_d = lambda col: oof[col].groupby(lab2.ref_date).rank(pct=True)
oof['LR+rule0']       = 0.5 * rk_d('LR') + 0.5 * rk_d('rule0 (-log_past_med)')
oof['LR+q75']         = 0.5 * rk_d('LR') + 0.5 * rk_d('rule q75 a=0.8')
oof['LGB+LR+rule0']   = (rk_d('LGB') + rk_d('LR') + rk_d('rule0 (-log_past_med)')) / 3
oof['LGB+LR+q75']     = (rk_d('LGB') + rk_d('LR') + rk_d('rule q75 a=0.8')) / 3
summarize(lab2, oof)

,전체평균,early평균,late평균,08-25
rule0 (-log_past_med),0.2966,0.2869,0.3084,0.4039
rule q75 a=0.8,0.3339,0.3271,0.3422,0.4259
LR,0.3517,0.3589,0.3429,0.4161
LGB,0.3402,0.3393,0.3414,0.3856
LR+rule0,0.3310,0.3313,0.3306,0.4201
LR+q75,0.3503,0.3508,0.3497,0.4363
LGB+LR+rule0,0.3387,0.3413,0.3356,0.4158
LGB+LR+q75,0.3526,0.3547,0.3501,0.4312


### 6.1 결과 (채널 GroupKFold, 20개 기준일)
- LR 단독 0.352이 최고. **LR + 기존 규칙(1회차 제출 방식)은 0.331로 LR 단독보다 낮음** — 08-25 한 날짜(blend>LR)와 반대 방향. 단일 날짜 판단의 위험을 보여줌.
- q75 규칙은 규칙으로는 개선(0.297→0.334)이지만 LR과 섞으면 LR 단독 수준.

### 6.2 LR 개선 실험: 정규화 C, 분위수 피처 추가, blend 가중치

In [19]:
LF = ['L_mean', 'L_med3', 'L_med5', 'L_med10', 'L_q25', 'L_q60', 'L_q75', 'L_q90', 'L_max']
for d_ in (lab2,):
    d_['q75_minus_med'] = d_.L_q75 - d_.log_past_med
    d_['q90_minus_med'] = d_.L_q90 - d_.log_past_med
    d_['max_minus_med'] = d_.L_max - d_.log_past_med
    d_['med3_minus_med'] = d_.L_med3 - d_.log_past_med
FEXT = FEATURES + LF + ['q75_minus_med', 'q90_minus_med', 'max_minus_med', 'med3_minus_med']

fns2 = {}
for C in [0.01, 0.03, 0.1, 0.3]:
    fns2[f'LR C={C}']      = lambda trn, tst, C=C: lr_fn(trn, tst, C)
    fns2[f'LR+분위수 C={C}'] = lambda trn, tst, C=C: lr_fn(trn, tst, C, FEXT)
oof2 = oof_predict(lab2, fns2)
summarize(lab2, oof2)

,전체평균,early평균,late평균,08-25
LR C=0.01,0.3608,0.3709,0.3485,0.4181
LR+분위수 C=0.01,0.3462,0.3485,0.3434,0.4313
LR C=0.03,0.3570,0.3659,0.3460,0.4190
LR+분위수 C=0.03,0.3427,0.3454,0.3394,0.4225
LR C=0.1,0.3517,0.3589,0.3429,0.4161
LR+분위수 C=0.1,0.3395,0.3427,0.3356,0.4151
LR C=0.3,0.3503,0.3570,0.3422,0.4141
LR+분위수 C=0.3,0.3361,0.3396,0.3319,0.4106


### 6.3 결과: 정규화 강할수록 좋음(C=0.01 최고 0.361), 분위수 피처 추가는 해로움(0.346).
### 6.4 더 낮은 C + 피처 하나씩 제거(leave-one-feature-out) ablation

In [20]:
fns3 = {f'LR C={C}': (lambda trn, tst, C=C: lr_fn(trn, tst, C)) for C in [0.001, 0.003, 0.01]}
oof3 = oof_predict(lab2, fns3)
print(summarize(lab2, oof3))

C0 = 0.01
base = summarize(lab2, oof_predict(lab2, {'base': lambda trn, tst: lr_fn(trn, tst, C0)}))
b = base.loc['base', '전체평균']
rows = []
for f in FEATURES:
    cols = [x for x in FEATURES if x != f]
    o = oof_predict(lab2, {'x': lambda trn, tst, cols=cols: lr_fn(trn, tst, C0, cols)})
    s = summarize(lab2, o).loc['x']
    rows.append((f, s['전체평균'], round(s['전체평균'] - b, 4), s['late평균']))
abl = pd.DataFrame(rows, columns=['제거한 피처', '전체평균', '변화(−면 도움되던 피처)', 'late평균']).sort_values('변화(−면 도움되던 피처)', ascending=False)
print('기준(C=0.01, 23개):', b)
abl

              전체평균  early평균  late평균   08-25
LR C=0.001  0.3575   0.3690  0.3435  0.4137
LR C=0.003  0.3616   0.3720  0.3488  0.4189
LR C=0.01   0.3608   0.3709  0.3485  0.4181


기준(C=0.01, 23개): 0.3608


,제거한 피처,전체평균,변화(−면 도움되던 피처),late평균
13,comment_rate,0.3652,0.0044,0.3537
20,avg_dur,0.3631,0.0023,0.3501
15,log_q90,0.3624,0.0016,0.3495
2,like_rate,0.3623,0.0015,0.3492
14,log_q10,0.3619,0.0011,0.3490
16,trend_slope,0.3619,0.0011,0.3497
10,has_game,0.3616,0.0008,0.3502
6,days_since_long,0.3612,0.0004,0.3491
19,n_long_28,0.3610,0.0002,0.3492
17,days_since_meas,0.3609,0.0001,0.3485


### 6.5 ablation 결과
- 빼면 좋아지는 피처: comment_rate(+0.0044), avg_dur, log_q90, like_rate, log_q10, trend_slope, has_game
- 빼면 나빠지는(도움) 피처: mom_7(−0.012), n_short_14, log_view_std, log_n_long_5d, mom_14, log_short_med_rel — 단기 모멘텀·변동성 계열
- 정규화는 C=0.003~0.01이 최적(0.361)

### 6.6 피처 세트 비교: 해로운 피처 제거(PRUNED) / 단기 모멘텀 보강(+MOM). **fold 분할 시드 2개**로 선택 편향 점검

In [21]:
DROP = ['comment_rate', 'avg_dur', 'log_q90', 'like_rate', 'log_q10', 'trend_slope', 'has_game']
PRUNED = [f for f in FEATURES if f not in DROP]
for d_ in (lab2,):
    d_['med5_minus_med'] = d_.L_med5 - d_.log_past_med
    d_['med10_minus_med'] = d_.L_med10 - d_.log_past_med
MOM_EXTRA = PRUNED + ['med3_minus_med', 'med5_minus_med']

sets6 = {'전체23': FEATURES, 'PRUNED': PRUNED, 'PRUNED+모멘텀(med3,med5)': MOM_EXTRA}
allres = []
for seed in [0, 1]:
    fns6 = {n: (lambda trn, tst, cols=c: lr_fn(trn, tst, 0.01, cols)) for n, c in sets6.items()}
    o = oof_predict(lab2, fns6, seed=seed)
    s = summarize(lab2, o); s['seed'] = seed
    allres.append(s)
res6 = pd.concat(allres)
print(res6.sort_index())
print()
print(res6.groupby(level=0)[['전체평균', 'early평균', 'late평균', '08-25']].mean().round(4))

                         전체평균  early평균  late평균   08-25  seed
PRUNED                 0.3754   0.3865  0.3618  0.4215     0
PRUNED                 0.3814   0.3945  0.3654  0.4101     1
PRUNED+모멘텀(med3,med5)  0.3769   0.3883  0.3629  0.4291     0
PRUNED+모멘텀(med3,med5)  0.3806   0.3925  0.3660  0.4127     1
전체23                   0.3608   0.3709  0.3485  0.4181     0
전체23                   0.3724   0.3823  0.3604  0.4000     1

                         전체평균  early평균  late평균   08-25
PRUNED                 0.3784   0.3905  0.3636  0.4158
PRUNED+모멘텀(med3,med5)  0.3788   0.3904  0.3644  0.4209
전체23                   0.3666   0.3766  0.3544  0.4091


### 6.7 결과: PRUNED(16개, C=0.01)가 전체23개보다 +0.012 (두 시드 모두, late 구간도 +0.009). 모멘텀 보강은 추가 이득 없음 → PRUNED 채택.
### 6.8 PRUNED LR에 q75 규칙 / LGB를 섞는 것이 이득인가 (시드 0,1 평균)

In [22]:
def lgb_pruned(trn, tst):
    return np.mean([gb(s).fit(trn[PRUNED], trn.target).predict_proba(tst[PRUNED])[:, 1] for s in range(3)], axis=0)

rows = []
for seed in [0, 1]:
    o = oof_predict(lab2, {
        'LR': lambda trn, tst: lr_fn(trn, tst, 0.01, PRUNED),
        'LGB': lgb_pruned,
        'q75': lambda trn, tst: rule_score(tst, 0.8).values,
    }, seed=seed)
    rk_ = lambda c: o[c].groupby(lab2.ref_date).rank(pct=True)
    o['LR 단독'] = o['LR']
    o['LGB 단독'] = o['LGB']
    for w in [0.15, 0.3, 0.5]:
        o[f'LR+q75 (규칙 {w})'] = (1 - w) * rk_('LR') + w * rk_('q75')
    for w in [0.3, 0.5]:
        o[f'LR+LGB (LGB {w})'] = (1 - w) * rk_('LR') + w * rk_('LGB')
    o['LR+LGB+q75 (0.5/0.25/0.25)'] = 0.5 * rk_('LR') + 0.25 * rk_('LGB') + 0.25 * rk_('q75')
    s = summarize(lab2, o.drop(columns=['LR', 'LGB', 'q75'])); s['seed'] = seed
    rows.append(s)
r68 = pd.concat(rows)
r68.groupby(level=0)[['전체평균', 'early평균', 'late평균', '08-25']].mean().round(4).sort_values('전체평균', ascending=False)

,전체평균,early평균,late평균,08-25
LR 단독,0.3784,0.3905,0.3636,0.4158
LR+q75 (규칙 0.15),0.3776,0.3882,0.3647,0.4333
LR+LGB (LGB 0.3),0.3763,0.3843,0.3665,0.4154
LR+LGB+q75 (0.5/0.25/0.25),0.3737,0.3802,0.3657,0.4332
LR+q75 (규칙 0.3),0.3730,0.3818,0.3621,0.4396
LR+LGB (LGB 0.5),0.3708,0.3762,0.3642,0.4116
LR+q75 (규칙 0.5),0.3651,0.3706,0.3585,0.4452
LGB 단독,0.3506,0.3521,0.3486,0.3886


### 6.9 결과: PRUNED LR 단독 0.3784이 전체평균 최고. 어떤 blend도 전체평균은 LR을 못 넘음(차이 ≤0.005, 노이즈 수준). 규칙을 아주 조금(0.15) 섞으면 전체는 동일하고 late/08-25가 소폭 상승.
### 6.10 기존 규칙(rule0)을 같은 조건으로 확인 + 제출 후보 생성

In [23]:
rows = []
for seed in [0, 1]:
    o = oof_predict(lab2, {
        'LR': lambda trn, tst: lr_fn(trn, tst, 0.01, PRUNED),
        'rule0': lambda trn, tst: rule0(tst).values,
        'q75': lambda trn, tst: rule_score(tst, 0.8).values,
    }, seed=seed)
    rk_ = lambda c: o[c].groupby(lab2.ref_date).rank(pct=True)
    for w in [0.15, 0.3]:
        o[f'LR+rule0 (규칙 {w})'] = (1 - w) * rk_('LR') + w * rk_('rule0')
        o[f'LR+q75 (규칙 {w})']   = (1 - w) * rk_('LR') + w * rk_('q75')
    s = summarize(lab2, o.drop(columns=['LR', 'rule0', 'q75'])); s['seed'] = seed
    rows.append(s)
pd.concat(rows).groupby(level=0)[['전체평균', 'early평균', 'late평균', '08-25']].mean().round(4)

,전체평균,early평균,late평균,08-25
LR+q75 (규칙 0.15),0.3776,0.3882,0.3647,0.4333
LR+q75 (규칙 0.3),0.3730,0.3818,0.3621,0.4396
LR+rule0 (규칙 0.15),0.3661,0.3736,0.3570,0.4230
LR+rule0 (규칙 0.3),0.3549,0.3591,0.3498,0.4247


## 7. 제출 후보 v3: PRUNED LR(C=0.01) + q75 규칙 0.15
- 학습: 자체 FUT26 07-26~08-05, 자체 FUT18 08-06~08-13, 공식 08-11·08-25 = lab2 전체(10,472행)
- 점수 = 0.85·rank(LR) + 0.15·rank(0.8·q75 − log_past_med)
- 근거: 채널 GroupKFold 20개 기준일에서 LR(PRUNED) 0.378 (이전 제출 방식 LR23+rule0 0.331), 규칙 소량 blend는 08-25/late 보강용

In [24]:
W3 = 0.15
final3 = logreg(0.01).fit(lab2[PRUNED], lab2.target)
test3 = (sample[['row_id', 'channel_id']]
         .merge(make_features(D), on='channel_id', how='left')
         .merge(level_feats(D), on='channel_id', how='left'))
proba3 = final3.predict_proba(test3[PRUNED])[:, 1]
test3['prediction'] = (1 - W3) * rank01(proba3) + W3 * rank01(rule_score(test3, 0.8))
sub3 = sample[['row_id']].merge(test3[['row_id', 'prediction']], on='row_id', how='left')

assert list(sub3.columns) == ['row_id', 'prediction'] and len(sub3) == len(sample) == 694
assert set(sub3.row_id) == set(sample.row_id) and sub3.row_id.is_unique
assert sub3.prediction.notna().all() and np.isfinite(sub3.prediction).all() and sub3.prediction.between(0, 1).all()
FILE3 = '../submissions/submit_06_lrpruned_c001_q75w015.csv'
sub3.to_csv(FILE3, index=False, encoding='utf-8')

s = sub3.set_index('row_id').prediction
for nm, f in [('submit_04 (1회차 0.4554)', '../submissions/submit_04_logreg_rankblend_fut26.csv'),
              ('submit_05 (2회차 0.4402)', '../submissions/submit_05_logreg_q75rule_a08.csv')]:
    o_ = pd.read_csv(f).set_index('row_id').prediction
    print(f'{nm} 과의 스피어만 상관: {s.corr(o_.loc[s.index], method="spearman"):.4f}')
print('저장:', FILE3)

submit_04 (1회차 0.4554) 과의 스피어만 상관: 0.9328
submit_05 (2회차 0.4402) 과의 스피어만 상관: 0.9380
저장: ../submissions/submit_06_lrpruned_c001_q75w015.csv


## 8. 앙상블 시험 (채널 GroupKFold, 20개 기준일, 시드 0·1 평균)
구성원: PRUNED LR(C=0.01) / 23개 LR / **배깅 LR**(채널 부트스트랩 + 피처 70% 샘플, 15개) / RandomForest·ExtraTrees(얕게, 리프 50) / LGB / q75 규칙. 모두 날짜 내 rank 평균으로 결합.

In [25]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

def bag_lr(trn, tst, n_bags=15, frac=0.7, C=0.01, seed=0):
    rng = np.random.default_rng(seed)
    chans = trn.channel_id.unique()
    preds = []
    for b in range(n_bags):
        boot = rng.choice(chans, size=len(chans), replace=True)
        sub = trn[trn.channel_id.isin(set(boot))]
        cols = list(rng.choice(PRUNED, size=int(len(PRUNED) * frac), replace=False))
        preds.append(rank01(logreg(C).fit(sub[cols], sub.target).predict_proba(tst[cols])[:, 1]))
    return np.mean(preds, axis=0)

def rf_fn(trn, tst, cls=RandomForestClassifier):
    imp = SimpleImputer(strategy='median').fit(trn[PRUNED])
    m = cls(n_estimators=300, min_samples_leaf=50, max_features=0.5, n_jobs=-1, random_state=0)
    return m.fit(imp.transform(trn[PRUNED]), trn.target).predict_proba(imp.transform(tst[PRUNED]))[:, 1]

rows = []
for seed in [0, 1]:
    o = oof_predict(lab2, {
        'LR': lambda trn, tst: lr_fn(trn, tst, 0.01, PRUNED),
        'LR23': lambda trn, tst: lr_fn(trn, tst, 0.01, FEATURES),
        'BAG': bag_lr,
        'RF': rf_fn,
        'ET': lambda trn, tst: rf_fn(trn, tst, ExtraTreesClassifier),
        'LGB': lgb_pruned,
        'q75': lambda trn, tst: rule_score(tst, 0.8).values,
    }, seed=seed)
    R = lambda c: o[c].groupby(lab2.ref_date).rank(pct=True)
    o['E1 LR+LR23+BAG']        = (R('LR') + R('LR23') + R('BAG')) / 3
    o['E2 LR+RF']              = (R('LR') + R('RF')) / 2
    o['E3 LR+ET']              = (R('LR') + R('ET')) / 2
    o['E4 LR+RF+ET+LGB']       = 0.4 * R('LR') + 0.2 * (R('RF') + R('ET') + R('LGB'))
    o['E5 BAG+RF+LGB']         = (R('BAG') + R('RF') + R('LGB')) / 3
    o['E6 LR+BAG+RF+q75(.1)']  = 0.35 * R('LR') + 0.3 * R('BAG') + 0.25 * R('RF') + 0.1 * R('q75')
    s = summarize(lab2, o.drop(columns=['q75'])); s['seed'] = seed
    rows.append(s)
r8 = pd.concat(rows)
r8.groupby(level=0)[['전체평균', 'early평균', 'late평균', '08-25']].mean().round(4).sort_values('전체평균', ascending=False)

,전체평균,early평균,late평균,08-25
LR,0.3784,0.3905,0.3636,0.4158
E6 LR+BAG+RF+q75(.1),0.3731,0.3762,0.3694,0.4356
E1 LR+LR23+BAG,0.3730,0.3812,0.3629,0.4181
E3 LR+ET,0.3703,0.3748,0.3647,0.4096
E4 LR+RF+ET+LGB,0.3696,0.3732,0.3652,0.4170
BAG,0.3668,0.3719,0.3604,0.4170
LR23,0.3666,0.3766,0.3544,0.4091
E2 LR+RF,0.3661,0.3686,0.3631,0.4244
E5 BAG+RF+LGB,0.3630,0.3650,0.3605,0.4160
ET,0.3516,0.3560,0.3460,0.3938


### 8.1 결과: 어떤 앙상블도 전체평균에서 LR 단독(0.378)을 못 넘음. 트리 계열이 LR보다 한참 낮아(RF 0.329, ET 0.352, LGB 0.351) 섞을수록 끌려 내려감. late/08-25에서만 소폭 앞서나 노이즈 수준.

## 9. 최근 영상 반영 개선 (기준일 직전 영상의 view_5d를 더 정교하게)
공지로 확인된 대로 09-01 직전 영상의 view_5d도 사용 가능. 현재 `mom_7`(7일 이내 중앙값)이 가장 도움이 된 피처라, 최근성 가중 피처를 추가해 비교한다.
- `ew3/ew7/ew14`: 지수 감쇠 가중 평균 log(1+view_5d) (반감기 3·7·14일) − log_past_med
- `last1`, `last3`: 가장 최근 1편 / 최근 3편 중앙값 − log_past_med
- `hot5`: 최근 5편 중 채널 과거 중앙값을 넘은 비율

In [26]:
def recent_feats(Dref):
    return con.sql(f"""
    WITH lv AS (
        SELECT channel_id, LN(1 + view_5d) AS l,
               DATE_DIFF('hour', published_at, TIMESTAMP '{Dref}') / 24.0 AS days_ago,
               ROW_NUMBER() OVER (PARTITION BY channel_id ORDER BY published_at DESC) AS rn
        FROM video_5d_views
        WHERE is_shorts = 0 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '{Dref}'
    ), med AS (SELECT channel_id, MEDIAN(l) AS m FROM lv GROUP BY 1)
    SELECT lv.channel_id,
           SUM(l * EXP(-days_ago / 3.0))  / SUM(EXP(-days_ago / 3.0))  AS ew3,
           SUM(l * EXP(-days_ago / 7.0))  / SUM(EXP(-days_ago / 7.0))  AS ew7,
           SUM(l * EXP(-days_ago / 14.0)) / SUM(EXP(-days_ago / 14.0)) AS ew14,
           MAX(l) FILTER (WHERE rn = 1) AS last1,
           MEDIAN(l) FILTER (WHERE rn <= 3) AS last3,
           AVG(CASE WHEN l > med.m THEN 1.0 ELSE 0.0 END) FILTER (WHERE rn <= 5) AS hot5
    FROM lv JOIN med USING (channel_id) GROUP BY 1
    """).df()

rf_ = pd.concat([recent_feats(d).assign(ref_date=d) for d in sorted(lab2.ref_date.unique())], ignore_index=True)
lab2 = lab2.drop(columns=[c for c in rf_.columns if c in lab2.columns and c not in ('channel_id', 'ref_date')]).merge(rf_, on=['channel_id', 'ref_date'], how='left')
for c in ['ew3', 'ew7', 'ew14', 'last1', 'last3']:
    lab2[c + '_d'] = lab2[c] - lab2.log_past_med
NEWS = {'ew3': ['ew3_d'], 'ew7': ['ew7_d'], 'ew14': ['ew14_d'], 'last1': ['last1_d'], 'last3': ['last3_d'], 'hot5': ['hot5'],
        'ew3+ew14': ['ew3_d', 'ew14_d'], 'last3+hot5': ['last3_d', 'hot5'], 'ew전체+last+hot': ['ew3_d', 'ew7_d', 'ew14_d', 'last1_d', 'last3_d', 'hot5']}
allr = []
for seed in [0, 1]:
    fns = {'PRUNED(기준)': (lambda trn, tst: lr_fn(trn, tst, 0.01, PRUNED))}
    for k, v in NEWS.items():
        fns['+' + k] = (lambda trn, tst, cols=PRUNED + v: lr_fn(trn, tst, 0.01, cols))
    s = summarize(lab2, oof_predict(lab2, fns, seed=seed)); s['seed'] = seed
    allr.append(s)
r9 = pd.concat(allr)
r9.groupby(level=0)[['전체평균', 'early평균', 'late평균', '08-25']].mean().round(4).sort_values('전체평균', ascending=False)

,전체평균,early평균,late평균,08-25
+ew3,0.3832,0.3944,0.3694,0.4122
+last3,0.3802,0.3929,0.3648,0.4194
+last3+hot5,0.3797,0.3921,0.3646,0.4189
+ew7,0.3794,0.3909,0.3654,0.4102
PRUNED(기준),0.3784,0.3905,0.3636,0.4158
+hot5,0.3783,0.3899,0.3641,0.4178
+last1,0.3764,0.3876,0.3628,0.4159
+ew14,0.3751,0.3853,0.3626,0.4154
+ew3+ew14,0.3746,0.3840,0.3629,0.4157
+ew전체+last+hot,0.3720,0.3813,0.3607,0.4167


### 9.1 결과: 최근성 가중 피처는 이득이 작음(+ew3 0.3784→0.3832, 나머지는 무효 또는 하락). 
### 9.2 데이터 범위 점검 — 기준일 이후 시점의 정보가 어디까지 들어 있는가

In [27]:
con.sql("CREATE OR REPLACE VIEW video_snapshots AS SELECT * FROM '../data/processed/video_snapshots.parquet'")
print(con.sql("SELECT MIN(published_at) mn, MAX(published_at) mx, COUNT(*) n FROM videos").df())
print(con.sql("SELECT MIN(collected_at) mn, MAX(collected_at) mx, COUNT(*) n, COUNT(DISTINCT video_id) v FROM video_snapshots").df())
print(con.sql("""SELECT DATE_TRUNC('day', collected_at) d, COUNT(*) n FROM video_snapshots
                 WHERE collected_at >= TIMESTAMP '2026-08-25' GROUP BY 1 ORDER BY 1""").df().to_string())
print(con.sql("SELECT MIN(published_at), MAX(published_at), COUNT(*) FROM video_5d_views").df())
print(con.sql("DESCRIBE video_5d_views").df()[['column_name']].T.to_string())

                   mn                  mx      n
0 2026-07-01 07:58:41 2026-08-31 23:59:27  38411
                   mn                  mx         n      v
0 2026-07-15 08:34:38 2026-08-31 23:45:54  18051339  37821
           d       n
0 2026-08-25  458447
1 2026-08-26  460591
2 2026-08-27  461458
3 2026-08-28  466856
4 2026-08-29  470285
5 2026-08-30  476655
6 2026-08-31  482763
    min(published_at)   max(published_at)  count_star()
0 2026-07-10 08:39:51 2026-08-31 23:59:27         31524


                    0           1             2        3        4           5         6        7        8          9
column_name  video_id  channel_id  published_at  view_5d  like_5d  comment_5d  h_before  h_after  max_gap  is_shorts


### 9.2 결과: `video_snapshots`/`videos`는 08-31까지만 존재. 09-01 이후 정보는 `video_5d_views`의 view_5d(이미 사용 중)뿐 → 기준일 직전 영상 정보는 이미 최대한 반영됨. 추가로 파낼 신호 없음.

## 10. 제출 후보 v4: PRUNED + ew3 LR(C=0.01) + q75 규칙 0.15
채널 GroupKFold(시드 0·1 평균)로 v3와 비교한 뒤 파일을 만든다.

In [28]:
F4 = PRUNED + ['ew3_d']
rows = []
for seed in [0, 1]:
    o = oof_predict(lab2, {'LR3': lambda trn, tst: lr_fn(trn, tst, 0.01, PRUNED),
                           'LR4': lambda trn, tst: lr_fn(trn, tst, 0.01, F4),
                           'q75': lambda trn, tst: rule_score(tst, 0.8).values}, seed=seed)
    R = lambda c: o[c].groupby(lab2.ref_date).rank(pct=True)
    o['v3: LR(PRUNED)+q75 .15'] = 0.85 * R('LR3') + 0.15 * R('q75')
    o['v4: LR(PRUNED+ew3)+q75 .15'] = 0.85 * R('LR4') + 0.15 * R('q75')
    s = summarize(lab2, o.drop(columns=['q75'])); s['seed'] = seed
    rows.append(s)
pd.concat(rows).groupby(level=0)[['전체평균', 'early평균', 'late평균', '08-25']].mean().round(4)

,전체평균,early평균,late평균,08-25
LR3,0.3784,0.3905,0.3636,0.4158
LR4,0.3832,0.3944,0.3694,0.4122
v3: LR(PRUNED)+q75 .15,0.3776,0.3882,0.3647,0.4333
v4: LR(PRUNED+ew3)+q75 .15,0.3808,0.3882,0.3720,0.4322


In [29]:
final4 = logreg(0.01).fit(lab2[F4], lab2.target)
test4 = (sample[['row_id', 'channel_id']]
         .merge(make_features(D), on='channel_id', how='left')
         .merge(level_feats(D), on='channel_id', how='left')
         .merge(recent_feats(D), on='channel_id', how='left'))
test4['ew3_d'] = test4.ew3 - test4.log_past_med
proba4 = final4.predict_proba(test4[F4])[:, 1]
test4['prediction'] = 0.85 * rank01(proba4) + 0.15 * rank01(rule_score(test4, 0.8))
sub4 = sample[['row_id']].merge(test4[['row_id', 'prediction']], on='row_id', how='left')

assert list(sub4.columns) == ['row_id', 'prediction'] and len(sub4) == len(sample) == 694
assert set(sub4.row_id) == set(sample.row_id) and sub4.row_id.is_unique
assert sub4.prediction.notna().all() and np.isfinite(sub4.prediction).all() and sub4.prediction.between(0, 1).all()
FILE4 = '../submissions/submit_07_lrpruned_ew3_q75w015.csv'
sub4.to_csv(FILE4, index=False, encoding='utf-8')
s4 = sub4.set_index('row_id').prediction
for nm, f in [('submit_04 (0.4554)', '../submissions/submit_04_logreg_rankblend_fut26.csv'),
              ('submit_05 (0.4402)', '../submissions/submit_05_logreg_q75rule_a08.csv'),
              ('submit_06 (미제출)', '../submissions/submit_06_lrpruned_c001_q75w015.csv')]:
    o_ = pd.read_csv(f).set_index('row_id').prediction
    print(f'{nm} 과의 스피어만 상관: {s4.corr(o_.loc[s4.index], method="spearman"):.4f}')
print('저장:', FILE4)

submit_04 (0.4554) 과의 스피어만 상관: 0.9159
submit_05 (0.4402) 과의 스피어만 상관: 0.9260
submit_06 (미제출) 과의 스피어만 상관: 0.9905
저장: ../submissions/submit_07_lrpruned_ew3_q75w015.csv


### 10.1 v4 결과: 20일 평균 0.3808 (v3 0.3776). 소폭 개선 — 더 이상 의미 없음.

## 11. 오라클 분석: 어디에 큰 개선 여지가 있는가
정답 구간(D~D+FUT)의 정보를 일부러 알려 주고 점수가 얼마나 뛰는지 본다. (제출용 아님, 진단용)
- `has_fut`: 정답 구간에 view_5d가 있는 롱폼이 1편 이상 있는가 (없으면 future=0 → 확정 비Rising)
- `log_n_fut`: 정답 구간 롱폼 수
- `fut_lvl`: 정답 구간 로그 중앙값 (완전 정답에 가까움)

In [30]:
FUT_OF = lambda d: 26 if d <= '2026-08-05' else 18
def fut_stats(Dref):
    fut = FUT_OF(Dref)
    return con.sql(f"""
    SELECT channel_id, COUNT(view_5d) AS n_fut, MEDIAN(LN(1 + view_5d)) AS fut_lvl
    FROM video_5d_views
    WHERE is_shorts = 0 AND view_5d IS NOT NULL
      AND published_at >= TIMESTAMP '{Dref}' AND published_at < TIMESTAMP '{Dref}' + INTERVAL {fut} DAY
    GROUP BY 1""").df()
fs = pd.concat([fut_stats(d).assign(ref_date=d) for d in sorted(lab2.ref_date.unique())], ignore_index=True)
lab2 = lab2.drop(columns=[c for c in ['n_fut', 'fut_lvl', 'has_fut', 'log_n_fut'] if c in lab2.columns]).merge(fs, on=['channel_id', 'ref_date'], how='left')
lab2['n_fut'] = lab2.n_fut.fillna(0)
lab2['has_fut'] = (lab2.n_fut > 0).astype(int)
lab2['log_n_fut'] = np.log1p(lab2.n_fut)
lab2['fut_lvl_d'] = lab2.fut_lvl.fillna(0) - lab2.log_past_med

print('정답 구간에 롱폼 업로드가 없는 채널 비율:', lab2.groupby('grp').has_fut.apply(lambda s: round(1 - s.mean(), 3)).to_dict())
print('has_fut=0 중 Rising 비율:', round(lab2[lab2.has_fut == 0].target.mean(), 4), '| has_fut=1 중:', round(lab2[lab2.has_fut == 1].target.mean(), 4))

orc = {
    '현재(PRUNED+ew3)':                   PRUNED + ['ew3_d'],
    '+오라클 has_fut':                    PRUNED + ['ew3_d', 'has_fut'],
    '+오라클 has_fut, log_n_fut':         PRUNED + ['ew3_d', 'has_fut', 'log_n_fut'],
    '+오라클 미래수준(fut_lvl_d)':         PRUNED + ['ew3_d', 'fut_lvl_d'],
}
o = oof_predict(lab2, {k: (lambda trn, tst, c=c: lr_fn(trn, tst, 0.01, c)) for k, c in orc.items()})
summarize(lab2, o)

정답 구간에 롱폼 업로드가 없는 채널 비율: {'early': 0.014, 'late': 0.049}
has_fut=0 중 Rising 비율: 0.0299 | has_fut=1 중: 0.2064


,전체평균,early평균,late평균,08-25
현재(PRUNED+ew3),0.3800,0.3890,0.3690,0.4179
+오라클 has_fut,0.3999,0.4071,0.3912,0.4695
"+오라클 has_fut, log_n_fut",0.4030,0.4112,0.3931,0.4609
+오라클 미래수준(fut_lvl_d),0.8456,0.8489,0.8416,0.7095


### 11.1 오라클 결과
- 정답 구간에 롱폼이 없는 채널은 early 1.4% / late 4.9%뿐 → 업로드 유무·개수를 알아도 +0.02 (업로드 예측은 큰 여지 아님)
- 미래 조회수 수준을 알면 0.85 → **유일한 큰 레버는 "미래 영상 조회수 수준"**
- 그런데 이 값은 영상 한 편마다 운의 영향이 크다. 현재 점수가 순수 샘플링 노이즈 한계에 얼마나 가까운지 시뮬레이션으로 본다.

### 11.2 천장 시뮬레이션
가정: 채널 진짜 수준 μ_c는 변하지 않고, 영상 로그 조회수 ~ N(μ_c, σ_c²). 실제 데이터의 (영상 수 n, σ, 정답 구간 영상 수 M, 채널 간 분산 τ²)로 과거/미래 중앙값을 생성해 라벨을 만들고, 모형을 아는 최적 예측기(사후 기반)의 PR-AUC를 계산한다.

In [31]:
rng = np.random.default_rng(0)
def simulate_ceiling(g, reps=60, noise_mult=1.57):
    n = g.n_long_5d.values.astype(float)
    s2p = np.nanmedian(g.log_view_std ** 2)
    s2 = ((n - 1) * g.log_view_std.fillna(np.sqrt(s2p)).values ** 2 + 3 * s2p) / (n - 1 + 3)
    M = np.maximum(g.n_fut.values, 1.0)
    l_obs = g.log_past_med.values
    m = l_obs.mean(); tau2 = max(l_obs.var() - np.mean(noise_mult * s2 / n), 0.05)
    w = tau2 / (tau2 + noise_mult * s2 / n)
    out = []
    for _ in range(reps):
        mu = rng.normal(m, np.sqrt(tau2), len(g))
        past = mu + rng.normal(0, np.sqrt(noise_mult * s2 / n))
        fut  = mu + rng.normal(0, np.sqrt(noise_mult * s2 / M))
        growth = fut - past
        y = (growth >= np.quantile(growth, 0.8)).astype(int)
        # 최적 예측기: 사후 μ|past 로 growth ~ N(μ̂ - past, v + noise_mult*s2/M), 임계값은 실제 분위수
        muhat = w * past + (1 - w) * m
        sd = np.sqrt(w * noise_mult * s2 / n + noise_mult * s2 / M)
        t = np.quantile(growth, 0.8)
        score = norm.sf((t - (muhat - past)) / sd)
        out.append(average_precision_score(y, score))
    return np.mean(out)

rows = []
for d, g in lab2.groupby('ref_date'):
    rows.append((d, round(simulate_ceiling(g), 4)))
sim = pd.DataFrame(rows, columns=['ref_date', '시뮬 천장(진짜수준 불변 가정)'])
sim['grp'] = np.where(sim.ref_date <= '2026-08-05', 'early', 'late')
print(sim.groupby('grp').mean(numeric_only=True).round(4))
print('전체 평균:', round(sim.iloc[:, 1].mean(), 4), '| 현재 모델 20일 평균 0.380 (late 0.369)')

       시뮬 천장(진짜수준 불변 가정)
grp                     
early             0.3833
late              0.3886
전체 평균: 0.3856 | 현재 모델 20일 평균 0.380 (late 0.369)


### 11.3 시뮬레이션 결과: 천장 0.386 (early 0.383 / late 0.389) vs 현재 모델 0.380 → 채널 이력만으로는 천장의 98%.

## 12. 구조 모형 신호를 LR 피처로 (마지막 모델 쪽 시도)
영상 수 n·변동성 σ에 따라 평균회귀 강도와 불확실성이 달라지는 비선형 관계를 LR이 못 잡을 수 있다. 기준일별로 경험적 베이즈를 계산해 `shrink_gap`(수축 기대 성장률), `sd`, `z_t=(shrink_gap−t)/sd`를 만들어 PRUNED+ew3에 추가한다.

In [32]:
def struct_feats(g, k=3, fut=26, noise_mult=1.57):
    n = g.n_long_5d.astype(float); l = g.log_past_med.astype(float)
    s2p = np.nanmedian(g.log_view_std ** 2)
    s2 = (((n - 1) * g.log_view_std.fillna(np.sqrt(s2p)) ** 2) + k * s2p) / (n - 1 + k)
    m = l.mean(); tau2 = max(l.var() - np.mean(s2 / n), 0.05)
    w = tau2 / (tau2 + s2 / n)
    gap = (1 - w) * (m - l)
    rate = (g.n_long_28.fillna(0) + 0.5) / 28
    M = np.maximum(rate * fut, 1.0)
    sd = np.sqrt(w * s2 / n + noise_mult * s2 / M)
    out = pd.DataFrame({'shrink_gap': gap, 'st_sd': sd, 'st_w': w}, index=g.index)
    for t in [0.0, 0.5, 1.0]:
        out[f'z_{t}'] = (gap - t) / sd
    return out

sf = pd.concat([struct_feats(g) for _, g in lab2.groupby('ref_date')])
lab2 = lab2.drop(columns=[c for c in sf.columns if c in lab2.columns]).join(sf)

S_ALL = ['shrink_gap', 'st_sd', 'st_w', 'z_0.0', 'z_0.5', 'z_1.0']
base = PRUNED + ['ew3_d']
sets12 = {'기준 PRUNED+ew3': base, '+z만': base + ['z_0.0', 'z_0.5', 'z_1.0'],
          '+gap,sd': base + ['shrink_gap', 'st_sd'], '+전부': base + S_ALL}
rows = []
for seed in [0, 1]:
    s = summarize(lab2, oof_predict(lab2, {k: (lambda trn, tst, c=c: lr_fn(trn, tst, 0.01, c)) for k, c in sets12.items()}, seed=seed))
    rows.append(s)
pd.concat(rows).groupby(level=0)[['전체평균', 'early평균', 'late평균', '08-25']].mean().round(4)

,전체평균,early평균,late평균,08-25
"+gap,sd",0.3896,0.4056,0.3702,0.3883
+z만,0.3715,0.3758,0.3662,0.4195
+전부,0.3820,0.3934,0.3681,0.4034
기준 PRUNED+ew3,0.3832,0.3944,0.3694,0.4122


### 12.1 결과: 구조 신호를 LR 피처로 넣어도 의미 있는 개선 없음 (기준 0.3832 → +gap,sd 0.3896이나 late 동일·08-25 하락, +z 0.3715, +전부 0.3820). 채택하지 않음.

## 13. 모델 쪽 결론
- 오라클: 정답 구간 업로드 유무·개수를 알아도 +0.02. 큰 레버는 미래 조회수 수준뿐이며 이는 영상별 운이 크다.
- 천장 시뮬레이션 0.386 vs 현재 0.380 → 채널 이력 기반으로는 천장에 근접.
- 구조 모형, 앙상블, 트리, 분위수/최근성 피처, 피처 축소는 모두 천장 안쪽의 소폭 변화.
- 최종 후보: submit_07 (PRUNED+ew3 LR C=0.01 + q75 규칙 0.15).

## 14. 리더보드 제출 후보 팩 (로컬 점수가 높은 순)
리더보드가 최종 점수이므로, 로컬 검증에서 높게 나온 서로 다른 변형을 만들어 제출 후 가장 높은 것을 고른다.
후보끼리 상관이 높아 점수 차는 대부분 노이즈이며, 이는 리더보드 694행에 맞추는 것임을 인지하고 사용한다.

| 이름 | 구성 |
|---|---|
| A | LR(PRUNED+ew3+gap,sd) 단독 |
| B | LR(PRUNED+ew3) 단독 |
| C | LR(PRUNED+ew3+gap,sd) 0.85 + q75 0.15 |
| D | LR(PRUNED+ew3) 0.7 + q75 0.3 |
| E | LR 0.35 + 배깅LR 0.3 + RF 0.25 + q75 0.1 (E6 변형) |
| (기존) submit_07 | LR(PRUNED+ew3) 0.85 + q75 0.15 |

In [33]:
FA = PRUNED + ['ew3_d', 'shrink_gap', 'st_sd']
FB = PRUNED + ['ew3_d']

def bag_lr2(trn, tst, cols, n_bags=15, frac=0.7, C=0.01, seed=0):
    rng = np.random.default_rng(seed); chans = trn.channel_id.unique(); preds = []
    for b in range(n_bags):
        boot = rng.choice(chans, size=len(chans), replace=True)
        sub = trn[trn.channel_id.isin(set(boot))]
        c = list(rng.choice(cols, size=int(len(cols) * frac), replace=False))
        preds.append(rank01(logreg(C).fit(sub[c], sub.target).predict_proba(tst[c])[:, 1]))
    return np.mean(preds, axis=0)

def rf_fn2(trn, tst, cols):
    imp = SimpleImputer(strategy='median').fit(trn[cols])
    m = RandomForestClassifier(n_estimators=300, min_samples_leaf=50, max_features=0.5, n_jobs=-1, random_state=0)
    return m.fit(imp.transform(trn[cols]), trn.target).predict_proba(imp.transform(tst[cols]))[:, 1]

def pack_scores(trn, tst):
    """trn/tst -> {후보: 점수} (날짜 내 rank 결합은 호출부에서)"""
    p = {'LA': lr_fn(trn, tst, 0.01, FA), 'LB': lr_fn(trn, tst, 0.01, FB),
         'BAG': bag_lr2(trn, tst, FB), 'RF': rf_fn2(trn, tst, FB), 'q75': rule_score(tst, 0.8).values}
    return p

cv_rows = []
for seed in [0, 1]:
    ch = lab2.channel_id.values; u = np.unique(ch); rg = np.random.default_rng(seed); rg.shuffle(u)
    fo = {c: i % 5 for i, c in enumerate(u)}; fold = np.array([fo[c] for c in ch])
    raw = {k: np.zeros(len(lab2)) for k in ['LA', 'LB', 'BAG', 'RF', 'q75']}
    for f in range(5):
        pr = pack_scores(lab2[fold != f], lab2[fold == f])
        for k in raw: raw[k][fold == f] = pr[k]
    raw = pd.DataFrame(raw, index=lab2.index)
    R = lambda c: raw[c].groupby(lab2.ref_date).rank(pct=True)
    cand = pd.DataFrame({
        'A  LR+gap,sd 단독': R('LA'),
        'B  LR 단독': R('LB'),
        'C  A 0.85 + q75 0.15': 0.85 * R('LA') + 0.15 * R('q75'),
        'D  B 0.7 + q75 0.3': 0.7 * R('LB') + 0.3 * R('q75'),
        'E  LR+BAG+RF+q75': 0.35 * R('LB') + 0.3 * R('BAG') + 0.25 * R('RF') + 0.1 * R('q75'),
        'submit_07  B 0.85 + q75 0.15': 0.85 * R('LB') + 0.15 * R('q75')}, index=lab2.index)
    cv_rows.append(summarize(lab2, cand))
cv_pack = pd.concat(cv_rows).groupby(level=0)[['전체평균', 'early평균', 'late평균', '08-25']].mean().round(4).sort_values('전체평균', ascending=False)
cv_pack

,전체평균,early평균,late평균,08-25
"A LR+gap,sd 단독",0.3896,0.4056,0.3702,0.3883
C A 0.85 + q75 0.15,0.3850,0.3946,0.3734,0.4131
B LR 단독,0.3832,0.3944,0.3694,0.4122
submit_07 B 0.85 + q75 0.15,0.3808,0.3882,0.3720,0.4322
E LR+BAG+RF+q75,0.3793,0.3782,0.3807,0.4414
D B 0.7 + q75 0.3,0.3746,0.3793,0.3688,0.4416


In [34]:
# 09-01 예측 파일 생성
test_f = test4.copy()
test_f = test_f.join(struct_feats(test_f))
fin = {'LA': logreg(0.01).fit(lab2[FA], lab2.target), 'LB': logreg(0.01).fit(lab2[FB], lab2.target)}
pA = fin['LA'].predict_proba(test_f[FA])[:, 1]
pB = fin['LB'].predict_proba(test_f[FB])[:, 1]
pq = rule_score(test_f, 0.8).values
# 배깅·RF는 전체 학습 후 예측
pbag = bag_lr2(lab2, test_f, FB)
prf = rf_fn2(lab2, test_f, FB)

cands = {
    'A': rank01(pA),
    'B': rank01(pB),
    'C': 0.85 * rank01(pA) + 0.15 * rank01(pq),
    'D': 0.7 * rank01(pB) + 0.3 * rank01(pq),
    'E': 0.35 * rank01(pB) + 0.3 * rank01(pbag) + 0.25 * rank01(prf) + 0.1 * rank01(pq),
}
names = {'A': 'submit_08_A_lr_gapsd.csv', 'B': 'submit_09_B_lr_ew3.csv', 'C': 'submit_10_C_lrgapsd_q75w015.csv',
         'D': 'submit_11_D_lr_q75w03.csv', 'E': 'submit_12_E_ens_lr_bag_rf_q75.csv'}
ref = pd.read_csv('../submissions/submit_04_logreg_rankblend_fut26.csv').set_index('row_id').prediction
out = []
for k, sc in cands.items():
    t = test_f[['row_id']].copy(); t['prediction'] = sc
    s_ = sample[['row_id']].merge(t, on='row_id', how='left')
    assert list(s_.columns) == ['row_id', 'prediction'] and len(s_) == 694 and s_.row_id.is_unique
    assert s_.prediction.notna().all() and np.isfinite(s_.prediction).all() and s_.prediction.between(0, 1).all()
    s_.to_csv('../submissions/' + names[k], index=False, encoding='utf-8')
    out.append((k, names[k], round(s_.set_index('row_id').prediction.corr(ref.loc[s_.row_id], method='spearman'), 4)))
pd.DataFrame(out, columns=['후보', '파일', 'submit_04(0.4554)와 순위상관'])

,후보,파일,submit_04(0.4554)와 순위상관
0,A,submit_08_A_lr_gapsd.csv,0.8648
1,B,submit_09_B_lr_ew3.csv,0.8823
2,C,submit_10_C_lrgapsd_q75w015.csv,0.9044
3,D,submit_11_D_lr_q75w03.csv,0.9400
4,E,submit_12_E_ens_lr_bag_rf_q75.csv,0.9013
